In [1]:
from pathlib import Path
import re
import pandas as pd

# Folder containing the importance-magnitude CSV files
input_folder = Path(".")

# Match files such as:
filename_pattern = re.compile(
    r"^indiv_importance_magnitude_cnn1_Cas9_(\d+)\.csv$"
)

# Detect matching files
matching_files = []

for file_path in input_folder.glob(
    "indiv_importance_magnitude_cnn1_Cas9_*.csv"
):
    match = filename_pattern.match(file_path.name)

    if match:
        model_number = int(match.group(1))
        matching_files.append((model_number, file_path))

# Sort by model number
matching_files.sort(key=lambda x: x[0])

print(f"Detected {len(matching_files)} importance-magnitude files:")

for model_number, file_path in matching_files:
    print(f"Model {model_number}: {file_path.name}")

if len(matching_files) == 0:
    raise FileNotFoundError(
        "No matching importance-magnitude CSV files were found."
    )

if len(matching_files) != 10:
    print(
        f"\nWarning: Expected 10 files, but detected "
        f"{len(matching_files)} files."
    )


# Read and combine all files
all_importance_tables = []

for model_number, file_path in matching_files:

    df = pd.read_csv(file_path)

    # Check that the expected columns exist
    required_columns = {"Feature", "Importance Magnitude"}

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"{file_path.name} does not contain the required columns: "
            f"'Feature' and 'Importance Magnitude'."
        )

    # Keep only the required columns
    df = df[["Feature", "Importance Magnitude"]].copy()

    # Convert importance values to numeric
    df["Importance Magnitude"] = pd.to_numeric(
        df["Importance Magnitude"],
        errors="coerce"
    )

    # Add model number for tracking
    df["Model Number"] = model_number

    all_importance_tables.append(df)

# Combine all model tables
combined_df = pd.concat(
    all_importance_tables,
    ignore_index=True
)

# Calculate average importance magnitude for each feature
average_importance = (
    combined_df
    .groupby("Feature", as_index=False)
    .agg(
        Average_Importance_Magnitude=(
            "Importance Magnitude",
            "mean"
        ),
        Number_of_Models=(
            "Importance Magnitude",
            "count"
        )
    )
)

# Sort from highest to lowest average importance
average_importance = average_importance.sort_values(
    by="Average_Importance_Magnitude",
    ascending=False,
    ignore_index=True
)

# Save the final table
output_file = (
    input_folder /
    "indiv_average_importance_magnitude_cnn1_Cas9.csv"
)

average_importance.to_csv(output_file, index=False)

print(f"\nSaved: {output_file.name}")
print("\nAverage importance magnitude table:")
print(average_importance)

Detected 10 importance-magnitude files:
Model 32: indiv_importance_magnitude_cnn1_Cas9_32.csv
Model 48: indiv_importance_magnitude_cnn1_Cas9_48.csv
Model 55: indiv_importance_magnitude_cnn1_Cas9_55.csv
Model 66: indiv_importance_magnitude_cnn1_Cas9_66.csv
Model 71: indiv_importance_magnitude_cnn1_Cas9_71.csv
Model 84: indiv_importance_magnitude_cnn1_Cas9_84.csv
Model 85: indiv_importance_magnitude_cnn1_Cas9_85.csv
Model 91: indiv_importance_magnitude_cnn1_Cas9_91.csv
Model 92: indiv_importance_magnitude_cnn1_Cas9_92.csv
Model 93: indiv_importance_magnitude_cnn1_Cas9_93.csv

Saved: indiv_average_importance_magnitude_cnn1_Cas9.csv

Average importance magnitude table:
     Feature  Average_Importance_Magnitude  Number_of_Models
0    Pos20_G                      0.014539                10
1    Pos18_C                      0.013638                10
2    Pos20_C                      0.008994                10
3    Pos19_T                      0.007794                10
4    Pos19_C         

In [7]:
# PAM-order adjustment (reverse the order of the first 20 bases)
import pandas as pd
import re

# Input and output files
input_file = "indiv_average_importance_magnitude_cnn1_Cas9.csv"
output_file = "indiv_PAM_adjusted_average_importance_magnitude_cnn1_Cas9.csv"

# Read the CSV file
df = pd.read_csv(input_file)

# Function to adjust positions 1–20 only
def adjust_feature_position(feature):
    feature = str(feature).strip()

    # Match feature names such as Pos20_G
    match = re.match(r"^Pos(\d+)_(.+)$", feature)

    if not match:
        # Preserve feature names that do not follow the expected format
        return feature

    position = int(match.group(1))
    feature_type = match.group(2)

    # Reverse numbering only for the first 20 positions
    if 1 <= position <= 20:
        adjusted_position = 21 - position
        return f"Pos{adjusted_position}_{feature_type}"

    # Keep positions greater than 20 unchanged
    return feature


# Apply conversion to the Feature column
df["Feature"] = df["Feature"].apply(adjust_feature_position)

# Save the adjusted CSV file
df.to_csv(output_file, index=False)

print(f"Saved: {output_file}")

Saved: indiv_PAM_adjusted_average_importance_magnitude_cnn1_Cas9.csv


In [9]:
import pandas as pd

# Read the average importance-magnitude CSV file
df = pd.read_csv("indiv_PAM_adjusted_average_importance_magnitude_cnn1_Cas9.csv")

# Extract position and base from feature names
extracted = df["Feature"].str.extract(r"^Pos(\d+)_([ACGT])$")

df["Position"] = pd.to_numeric(extracted[0], errors="coerce")
df["Base"] = extracted[1]

# Ensure importance magnitudes are numeric
df["Average_Importance_Magnitude"] = pd.to_numeric(
    df["Average_Importance_Magnitude"],
    errors="coerce"
)

# Remove rows that don't match the expected format
df = df.dropna(
    subset=["Position", "Base", "Average_Importance_Magnitude"]
).copy()

df["Position"] = df["Position"].astype(int)

# Define windows (using actual positions)
windows = {
    "First_6": (1, 6),
    "Middle_8": (7, 14),
    "Last_6": (15, 20)
}

bases = ["A", "C", "G", "T"]

results = []

for base in bases:
    base_df = df[df["Base"] == base]

    row = {"Base": base}

    for window_name, (start, end) in windows.items():
        avg = base_df.loc[
            base_df["Position"].between(start, end),
            "Average_Importance_Magnitude"
        ].mean()

        row[window_name] = avg

    results.append(row)

# Convert to DataFrame
result_df = pd.DataFrame(results)

# Save as CSV
result_df.to_csv(
    "indiv_PAM_adjusted_average_importance_magnitude_by_Base_position_range_cnn1_Cas9.csv",
    index=False
)

print(result_df)

  Base   First_6  Middle_8    Last_6
0    A  0.003826  0.002524  0.002063
1    C  0.006731  0.002691  0.003447
2    G  0.005905  0.002483  0.002662
3    T  0.005188  0.002569  0.002371
